# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download

In [2]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [4]:
required_columns = [
    "content_hash_id",
    "client_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "engagement_rate",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "avg_engagement_sec",
    "position_bucket",
    "ga4_total_engagement_sec"
]

df = pd.read_parquet(
    parquet_path,
    columns=required_columns
)

print("Shape:", df.shape)







Shape: (9841378, 8)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [5]:
# CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Engagement rate
df["engagement_rate"] = np.where(
    df["ga4_sessions"] > 0,
    df["ga4_engaged_sessions"] / df["ga4_sessions"],
    0
)

# Average engagement time
df["avg_engagement_sec"] = np.where(
    df["ga4_sessions"] > 0,
    df["ga4_total_engagement_sec"] / df["ga4_sessions"],
    0
)

# Position bucket
df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=False
)

print("Engineered features created.")

Engineered features created.


In [6]:
required_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "engagement_rate",
    "avg_engagement_sec",
    "position_bucket"
]

missing = [
    col for col in required_features
    if col not in df.columns
]

print("Missing features:", missing)



Missing features: []


In [7]:
selected_features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ctr",
    "engagement_rate",
    "avg_engagement_sec",
    "position_bucket"
]

print("Selected features:")
print(selected_features)

Selected features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr', 'engagement_rate', 'avg_engagement_sec', 'position_bucket']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        groups=df["client_hash_id"]
    )
)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

Train rows: 8935676
Test rows: 905702


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
X_train = df.iloc[train_idx][selected_features]
X_test = df.iloc[test_idx][selected_features]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (8935676, 7)
X_test: (905702, 7)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
# Error analysis

error_df = pd.DataFrame({
    "actual_score": y_test,
    "predicted_score": predictions
})

error_df["absolute_error"] = (
    error_df["actual_score"]
    - error_df["predicted_score"]
).abs()

print("Largest prediction errors:")
print(
    error_df
    .sort_values("absolute_error", ascending=False)
    .head(10)
)


NameError: name 'y_test' is not defined

In [11]:
# Feature importance

feature_importance = pd.DataFrame({
    "Feature": selected_features,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

print("Feature importance:")
print(feature_importance)

NameError: name 'model' is not defined

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.